# Dataset preparation pipeline
1. Source dataset (raw images + masks)
2. Image-level split into train/val/test
3. Patch extraction (tiling)
4. Training using ImageFolder

This workflow prevents data leakage between splits.

## Imports & Reproducibility

In [1]:
import os
import torch
import shutil
import random
import numpy as np
from tqdm import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split

## Seed for reproducibility (BEST PRACTICE)

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## Data Preparation

### Image level split (train - val - test)

In [ ]:
# Define paths
raw_dir = Path("/home/adjalil/Working/data_lionel/data_src/raw")
mask_dir = Path("/home/adjalil/Working/data_lionel/data_src/mask")
output_dir = Path("/home/adjalil/Working/data_lionel/data_split") 

# Define val and test sizes
test_size = 0.15  # 15%
val_size = 0.15
random_seed = 42

# Your class names
classes = ["healthy", "mildiou"]

# ==================

all_images = []
all_labels = []

for cls in classes:
    class_images = list((raw_dir / cls).glob("*.tif"))
    for img_path in class_images:
        all_images.append(img_path)
        all_labels.append(cls)

print("Total images:", len(all_images))


# ===== Split train+val / test =====
train_val_imgs, test_imgs, train_val_labels, test_labels = train_test_split(
    all_images,
    all_labels,
    test_size=test_size,
    stratify=all_labels,
    random_state=set_seed(random_seed)
)

# ===== Split train / val =====
train_imgs, val_imgs, train_labels, val_labels = train_test_split(
    train_val_imgs,
    train_val_labels,
    test_size=val_size / (1 - test_size),
    stratify=train_val_labels,
    random_state=set_seed(random_seed)
)

print("Train:", len(train_imgs))
print("Val:", len(val_imgs))
print("Test:", len(test_imgs))


def copy_split(images_list, split_name):

    for img_path in images_list:

        cls = img_path.parent.name
        file_name = img_path.name

        dst_img_dir = output_dir / split_name / "raw" / cls
        dst_mask_dir = output_dir / split_name / "mask" / cls

        dst_img_dir.mkdir(parents=True, exist_ok=True)
        dst_mask_dir.mkdir(parents=True, exist_ok=True)

        # copy image
        shutil.copy(img_path, dst_img_dir / file_name)

        # build mask name 
        stem = img_path.stem
        mask_name = f"{stem}.tif"

        mask_path = mask_dir / cls / mask_name

        if mask_path.exists():
            shutil.copy(mask_path, dst_mask_dir / mask_name)
        else:
            print(f"[WARNING] Mask not found for {dst_img_dir / file_name}")


copy_split(train_imgs, "train")
copy_split(val_imgs, "val")
copy_split(test_imgs, "test")

print("Split finished !")

Total images: 28
Train: 18
Val: 5
Test: 5
Split finished !


### Patch

In [63]:
import numpy as np
import cv2
from pathlib import Path
import tifffile as tiff

def extract_classification_patches(
    images_dir,
    masks_dir,
    patch_size=128,
    use_ratio=True,
    threshold=0.05,
    classes = ["healthy", "diseased"],
    output_dir="dataset"
):

    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir)
    output_dir = Path(output_dir)

    clss_h = classes[0]
    clss_d = classes[1]

    healthy_dir = output_dir / clss_h
    diseased_dir = output_dir / clss_d

    healthy_dir.mkdir(parents=True, exist_ok=True)
    diseased_dir.mkdir(parents=True, exist_ok=True)

    sudirname = ['train', 'val', 'test']

    total_healthy = 0
    total_diseased = 0

    for cls in classes:

        image_files = list((images_dir / cls).glob("*.tif"))

        for c, image_file in enumerate(image_files):
            split = image_file.parents[2].name

            mask_file = masks_dir / cls / image_file.name
            print(f'[{c+1} / {len(image_files)} in {split}] Image :{os.path.basename(image_file)} -- Mask :{os.path.basename(mask_file)}')
        
            if not mask_file.exists():
                continue

            img = tiff.imread(str(image_file))
            mask = tiff.imread(str(mask_file))


            H, W = mask.shape
            n_rows = H // patch_size
            n_cols = W // patch_size

            for i in range(n_rows):
                for j in range(n_cols):

                    y0, x0 = i * patch_size, j * patch_size
                    y1, x1 = y0 + patch_size, x0 + patch_size

                    img_patch = img[y0:y1, x0:x1]
                    mask_patch = mask[y0:y1, x0:x1]
                    

                    ratio = (mask_patch > 0).mean()

                    is_diseased = ratio > (threshold if use_ratio else 0)

                    if is_diseased:
                        label_dir = diseased_dir
                        total_diseased += 1
                    else:
                        label_dir = healthy_dir
                        total_healthy += 1

                    name = f"{image_file.stem}_{i:03d}_{j:03d}.png"

                    img_rgb = cv2.cvtColor(img_patch, cv2.COLOR_BGR2RGB)
                    cv2.imwrite(str(label_dir / name), img_rgb)
                

    print("Healthy:", total_healthy)
    print("Mildiou:", total_diseased)




In [ ]:
# to execute
# train
extract_classification_patches(
    images_dir="/home/adjalil/Working/data_lionel/data_split/train/raw/", 
    masks_dir="/home/adjalil/Working/data_lionel/data_split/train/mask",  
    output_dir="/home/adjalil/Working/data_lionel/data_patch/train",
    patch_size=224,
    threshold=0.01,
    classes = ["healthy", "mildiou"]
)


# val
extract_classification_patches(
    images_dir="/home/adjalil/Working/data_lionel/data_split/val/raw",
    masks_dir="/home/adjalil/Working/data_lionel/data_split/val/mask",
    output_dir="/home/adjalil/Working/data_lionel/data_patch/val",
    patch_size=224,
    threshold=0.01,
    classes=["healthy","mildiou"]
)


# test
extract_classification_patches(
    images_dir="/home/adjalil/Working/data_lionel/data_split/test/raw/",
    masks_dir="/home/adjalil/Working/data_lionel/data_split/test/mask",
    output_dir="/home/adjalil/Working/data_lionel/data_patch/test",
    patch_size=224,
    threshold=0.01,
    classes=["healthy","mildiou"]
)


[1 / 9 in train] Image :uplot_2006002_1_camera_1_2_RGB_WB_V_U_1719300972.tif -- Mask :uplot_2006002_1_camera_1_2_RGB_WB_V_U_1719300972.tif
[2 / 9 in train] Image :uplot_109001_1_camera_1_5_RGB_WB_V_U_1723524789.tif -- Mask :uplot_109001_1_camera_1_5_RGB_WB_V_U_1723524789.tif
[3 / 9 in train] Image :uplot_2002002_1_camera_2_2_RGB_WB_V_U_1755511089.tif -- Mask :uplot_2002002_1_camera_2_2_RGB_WB_V_U_1755511089.tif
[4 / 9 in train] Image :uplot_2023002_1_camera_1_1_RGB_WB_V_U_1719302719.tif -- Mask :uplot_2023002_1_camera_1_1_RGB_WB_V_U_1719302719.tif
[5 / 9 in train] Image :uplot_2023002_1_camera_1_4_RGB_WB_V_U_1719302719.tif -- Mask :uplot_2023002_1_camera_1_4_RGB_WB_V_U_1719302719.tif
[6 / 9 in train] Image :uplot_2020002_1_camera_1_7_RGB_WB_V_U_1719302719.tif -- Mask :uplot_2020002_1_camera_1_7_RGB_WB_V_U_1719302719.tif
[7 / 9 in train] Image :uplot_2009002_1_camera_1_4_RGB_WB_V_U_1750660102.tif -- Mask :uplot_2009002_1_camera_1_4_RGB_WB_V_U_1750660102.tif
[8 / 9 in train] Image :uplot